In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import os

# ------------------------------
# CONFIGURATION
# ------------------------------

# runs = ["05", "06", "07", "08"]

file_pattern = "relative_uplift_timeseries_{}.csv"
depth_limit = 0.15         # meters
fly_out_value = 0.151       # value to assign once a block flies out
half_block_value = 0.0     # value to assign to half blocks (odd columns)

output_suffix = "_corrected.csv"

# ------------------------------
# CORRECT CSVs
# ------------------------------

for run in runs:
    csv_path = file_pattern.format(run)
    corrected_path = csv_path.replace(".csv", output_suffix)

    print(f"\n=== Correcting {csv_path} ===")

    # Load CSV
    df = pd.read_csv(csv_path)

    # Detect time columns automatically
    if df.columns[0].lower() == "step":
        time_cols = list(df.columns[:2])
        data_cols = list(df.columns[2:])
    else:
        time_cols = list(df.columns[:1])
        data_cols = list(df.columns[1:])

    # --------------------------------------------------
    # Calculate cumulative time if Step and StepTime exist
    # --------------------------------------------------

    if df.columns[0].lower() == "step":
        step_col = df.columns[0]
        step_time_col = df.columns[1]

        df[step_time_col] = pd.to_numeric(df[step_time_col], errors="coerce")

        cumulative_time = []
        time_offset = 0.0

        for step_name, group in df.groupby(step_col, sort=False):
            times = group[step_time_col]
            cumulative_times = times + time_offset
            cumulative_time.extend(cumulative_times)
            time_offset = cumulative_times.iloc[-1]

        df["CumulativeTime"] = cumulative_time

    # --------------------------------------------------
    # Check and correct uplift data
    # --------------------------------------------------

    data_values = df[data_cols].values.copy()

    # Boolean mask: True → block has flown out
    has_flown_out = np.zeros(len(data_cols), dtype=bool)

    for i in range(data_values.shape[0]):
        for j in range(data_values.shape[1]):
            if has_flown_out[j]:
                data_values[i, j] = fly_out_value
            else:
                val = data_values[i, j]
                if not np.isnan(val) and val > depth_limit:
                    has_flown_out[j] = True
                    data_values[i, j] = fly_out_value

    # --------------------------------------------------
    # Replace half blocks (odd-numbered columns)
    # --------------------------------------------------

    # Note: enumerate starting from 1 because user considers col 1, 3, 5 etc. as odd
    for idx, col in enumerate(data_cols, start=1):
        if idx % 2 == 1:
            # Overwrite entire column for half blocks
            data_values[:, idx - 1] = half_block_value

    # Create corrected DataFrame
    corrected_parts = [df[time_cols]]

    if "CumulativeTime" in df.columns:
        corrected_parts.append(df[["CumulativeTime"]])

    corrected_parts.append(pd.DataFrame(data_values, columns=data_cols))

    df_corrected = pd.concat(corrected_parts, axis=1)

    # Save corrected file
    df_corrected.to_csv(corrected_path, index=False)
    print(f"✅ Saved corrected file: {corrected_path}")


In [ ]:
# -*- coding: utf-8 -*-
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from scipy.linalg import solve
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Settings ---
fly_out_value = 0.151
target_time = 12   # time in seconds to visualize
csv_path = 'relative_uplift_timeseries_06_corrected.csv'
df_uplift = pd.read_csv(csv_path)

# ------------------------------
# Define brick layout parameters
# ------------------------------
full_w = 0.5
full_h = 0.5
half_w = 0.225
half_h = 0.5
horizontal_spacing = 0.05
v_spacing = 0.0

grid_width = (9 * full_w) + (8 * horizontal_spacing)
print(f"Grid width: {grid_width:.3f} m")

rows = 22

# ------------------------------
# Load uplift data
# ------------------------------

# Detect time column automatically
if df_uplift.columns[0].lower() == "step":
    time_col = "CumulativeTime" if "CumulativeTime" in df_uplift.columns else df_uplift.columns[1]
else:
    time_col = df_uplift.columns[0]

# Round times for matching
df_uplift[time_col] = df_uplift[time_col].round(2)

# Extract data for plotting the block grid
row_df = df_uplift[df_uplift[time_col] == round(target_time, 2)]
if row_df.empty:
    raise ValueError(f"⛔ Time step {target_time:.2f} not found in CSV.")

block_cols = [c for c in df_uplift.columns if c.startswith("B-")]
block_data = row_df[block_cols].iloc[0]

# Normalize color for uplift values
norm = plt.Normalize(vmin=0, vmax=0.015)

# Custom colormap
colors = ["#10b151", "#f6eb01", "#ffc112", "#ed2024"]
cmap = LinearSegmentedColormap.from_list("uplift_cmap", colors, N=256)
no_data_color = "lightgray"

# ------------------------------
# Simulate residual profiles
# ------------------------------

Hs = 0.6
rho_w = 1025.0
g = 9.81
Tp = 5.59
tan_beta = 1.0 / 3.0
D = 0.30
b = 0.30
k = 0.19
k_prime = 0.05
Lambda = np.sqrt(D * b * k / k_prime)
SWL = 4.7
dx = 0.01
x = np.arange(0, 20.0 + dx, dx)
N = len(x)

x_mwl = SWL / tan_beta
print(f"MWL (along slope): {x_mwl:.2f} m")

block_length = 0.5
slope = tan_beta
block_dx = block_length * np.cos(np.arctan(slope))
block_start = 6.0
block_positions = [block_start + i * block_dx for i in range(22)]

sigma = Hs / np.sqrt(2)
exceed_probs = np.arange(1, 6, 1) / 1000.0
wave_heights = sigma * np.sqrt(-2 * np.log(exceed_probs))

wave_pressures = {}
residual_profiles = {}
impact_centers = {}

for i, H_wave in enumerate(wave_heights):
    Tm_1_0 = Tp / 1.1
    L_m_1_0 = (g * Tm_1_0 ** 2) / (2.0 * np.pi)
    xi = tan_beta / np.sqrt(H_wave / L_m_1_0)
    z_SWL = H_wave * (0.8 + 0.6 * np.tanh(xi - 2.1)) if xi < 3 else np.nan
    x_center = (SWL - z_SWL) / tan_beta

    w_base = (7 / 6) * H_wave
    w_top = (1 / 3) * H_wave
    x_left_base = x_center - w_base / 2
    x_right_base = x_center + w_base / 2
    x_left_top = x_center - w_top / 2
    x_right_top = x_center + w_top / 2

    denom = (xi - 0.2) ** 2
    P_dimless = 8.0 - 1.6 * xi - 2.0 / denom
    P_max = P_dimless * rho_w * g * H_wave

    p_vals = np.zeros_like(x)
    for j, xx in enumerate(x):
        if x_left_base <= xx < x_left_top:
            p_vals[j] = P_max * (xx - x_left_base) / (x_left_top - x_left_base)
        elif x_left_top <= xx <= x_right_top:
            p_vals[j] = P_max
        elif x_right_top < xx <= x_right_base:
            p_vals[j] = P_max * (x_right_base - xx) / (x_right_base - x_right_top)

    b_vec = -p_vals.copy()
    b_vec[0] = b_vec[-1] = 0
    A = np.zeros((N, N))
    coeff = Lambda ** 2 / dx ** 2
    for j in range(1, N - 1):
        A[j, j - 1] = coeff
        A[j, j] = -2 * coeff - 1
        A[j, j + 1] = coeff
    A[0, 0] = A[-1, -1] = 1.0
    phi_F = solve(A, b_vec)
    residual = phi_F - p_vals

    wave_name = f"Wave {i+1} in 1000, H={H_wave:.2f}"
    residual_profiles[wave_name] = residual
    wave_pressures[wave_name] = p_vals
    impact_centers[wave_name] = x_center

wave_to_plot = list(residual_profiles.keys())[0]

active_wave = None
total_waves = 5
pulse_interval = 2.0
gravity_time = 2.0

for wave_idx in range(total_waves):
    t_start = gravity_time + wave_idx * pulse_interval
    t_end = gravity_time + (wave_idx + 1) * pulse_interval
    if t_start <= target_time < t_end:
        active_wave = wave_idx
        break

if active_wave is not None:
    wave_name = list(residual_profiles.keys())[active_wave]
    print(f"✅ Time {target_time:.2f} s belongs to {wave_name}")
else:
    print(f"ℹ️ Time {target_time:.2f} s is during gravity loading or after waves.")

# ------------------------------
# Plotting
# ------------------------------

fig = plt.figure(figsize=(10, 13))

gs = gridspec.GridSpec(
    nrows=2,
    ncols=2,
    width_ratios=[3, 1],
    height_ratios=[6, 1],
    wspace=0.05,
    hspace=0.3
)

# --- Top left: block grid ---
ax_grid = fig.add_subplot(gs[0, 0])

for row_num in range(1, rows + 1):
    current_y = (rows - row_num) * (full_h + v_spacing)
    is_half = (row_num % 2 == 1)
    if is_half:
        block_sequence = (
            [(half_w, f"B-{row_num}-1")] +
            [(full_w, f"B-{row_num}-{col}") for col in range(2, 10)] +
            [(half_w, f"B-{row_num}-10")]
        )
    else:
        block_sequence = [
            (full_w, f"B-{row_num}-{col}") for col in range(1, 10)
        ]

    x_pos = 0.0
    for block_w, block_name in block_sequence:
        if block_name in block_data:
            value = block_data[block_name]
            if np.isnan(value):
                color = no_data_color
            else:
                color = cmap(norm(value))
        else:
            color = no_data_color

        rect = patches.Rectangle(
            (x_pos, current_y),
            block_w, full_h,
            edgecolor='black',
            facecolor=color,
            linewidth=1
        )
        ax_grid.add_patch(rect)
        x_pos += block_w + horizontal_spacing

ax_grid.set_aspect('equal')
ax_grid.set_xlim(0, grid_width)
ax_grid.set_ylim(0, rows * full_h)
ax_grid.axis('off')

# Colorbar (taller and higher)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cax = fig.add_axes([-0.02, 0.5, 0.02, 0.4])
cbar = plt.colorbar(sm, cax=cax, orientation='vertical')
cbar.set_label('Relative Uplift [m]')
cbar.ax.yaxis.set_label_position('left')
cbar.ax.yaxis.set_ticks_position('left')

# --- Top right: residual pressure profile ---
ax_side = fig.add_subplot(gs[0, 1])

if active_wave is not None:
    wave_name = list(residual_profiles.keys())[active_wave]
    residual_vals = residual_profiles[wave_name]
    x_center = impact_centers[wave_name]
else:
    wave_name = wave_to_plot
    residual_vals = residual_profiles[wave_name]
    x_center = impact_centers[wave_name]

residual_vals_kPa = residual_vals / 1000.0

ax_side.plot(
    residual_vals_kPa,
    x,
    color='#1f77b4',
    linewidth=2,
    label=f"Residual Pressure\n{wave_name}"
)
ax_side.axhline(
    x_mwl,
    color='darkblue',
    linestyle='-',
    linewidth=2,
    label='MWL'
)
ax_side.axhline(
    x_center,
    color='black',
    linestyle='--',
    linewidth=2,
    label='Impact Center'
)
ax_side.invert_yaxis()
ax_side.set_xlabel('Pressure [kPa]')
ax_side.set_ylabel('x along slope [m]')
ax_side.set_ylim(5.5, 16.5)
ax_side.yaxis.tick_right()
ax_side.yaxis.set_label_position("right")
ax_side.set_title("Residual Pressure Profile")
ax_side.grid(True)
ax_side.legend(loc='lower right', fontsize=8)

# --- Bottom: max uplift over time ---
ax_uplift = fig.add_subplot(gs[1, :])

block_data_all = df_uplift[block_cols].copy()

fly_out_mask = (block_data_all == fly_out_value)
block_data_cleaned = block_data_all.replace(fly_out_value, np.nan)

max_uplift = block_data_cleaned.max(axis=1)

first_loss_times = []
for col in block_cols:
    fly_out_indices = fly_out_mask.index[fly_out_mask[col]].tolist()
    if fly_out_indices:
        first_idx = fly_out_indices[0]
        first_loss_times.append(df_uplift.loc[first_idx, time_col])

first_loss_times = sorted(set(first_loss_times))

ax_uplift.plot(
    df_uplift[time_col],
    max_uplift,
    color='navy',
    linewidth=2,
    label='Max Uplift'
)
ax_uplift.axvline(
    target_time,
    color='red',
    linestyle='-',
    linewidth=3,
    label=f"t = {target_time:.2f} s"
)

if len(first_loss_times) > 0:
    cross_y_value = max_uplift.max() * 1.1 if max_uplift.max() > 0 else 0.01
    ax_uplift.scatter(
        first_loss_times,
        [cross_y_value] * len(first_loss_times),
        color='red',
        marker='x',
        s=80,
        label='Block Loss (First Occurrence)'
    )

ax_uplift.set_xlabel("Time [s]")
ax_uplift.set_xlim(0, 12.1)
ax_uplift.set_ylabel("Max Relative Uplift [m]")
ax_uplift.set_title("Maximum Relative Uplift Over Time (All Blocks)")
ax_uplift.legend(loc='upper right', fontsize=8)
ax_uplift.grid(True)

# --- Global title ---
fig.suptitle(
    f"Significant Wave Height Hs = {Hs:.2f} m   |   Time = {target_time:.2f} s",
    fontsize=16,
    fontweight="bold"
)

plt.subplots_adjust(left=0, right=1, top=0.95, bottom=0, wspace=0, hspace=0)
plt.show()

fig.savefig(csv_path.replace(".csv", "_no_time.png"), dpi=300, bbox_inches='tight')
print("✅ Finished plotting block grid with uplift and residual pressure.")


In [1]:
# -*- coding: utf-8 -*-
# Batch plot "grid + residual + max uplift" for every per-job CSV (Py 2.7)

import os
import re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy.linalg import solve

# -----------------------------
# User settings
# -----------------------------
target_time = 12.0        # seconds to visualize (nearest match will be used)
fly_out_value = 0.151     # treat as missing

# If empty, uses the folder of this script:
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except:
    SCRIPT_DIR = os.getcwd()

# Pattern that matches the per-job CSVs we produced earlier:
CSV_SUFFIX = '_uplift_downdrift_ALLSTEPS_ALLFRAMES.csv'

# -----------------------------
# Brick layout parameters
# -----------------------------
full_w = 0.5
full_h = 0.5
half_w = 0.225
half_h = 0.5
horizontal_spacing = 0.05
v_spacing = 0.0
rows = 22

grid_width = (9 * full_w) + (8 * horizontal_spacing)
print("Grid width: %.3f m" % grid_width)

# Color map for uplift
colors = ["#10b151", "#f6eb01", "#ffc112", "#ed2024"]
cmap = LinearSegmentedColormap.from_list("uplift_cmap", colors, N=256)
no_data_color = "lightgray"
norm = plt.Normalize(vmin=0.0, vmax=0.015)

# -----------------------------
# Residual profile constants (same physics as before)
# -----------------------------
rho_w = 1025.0
g = 9.81
Tp = 5.59
tan_beta = 1.0 / 3.0
D = 0.30
b = 0.30
k = 0.19
k_prime = 0.05
Lambda = np.sqrt(D * b * k / k_prime)
SWL = 4.7
dx = 0.01
x = np.arange(0.0, 20.0 + dx, dx)
N = len(x)
x_mwl = SWL / tan_beta
block_length = 0.5
slope = tan_beta
block_dx = block_length * np.cos(np.arctan(slope))
block_start = 6.0
block_positions = [block_start + i * block_dx for i in range(22)]

# Time placement of waves (to map target_time to an active wave index)
total_waves = 5
pulse_interval = 2.0
gravity_time = 2.0

# -----------------------------
# Helpers
# -----------------------------
def extract_hs_from_job(job_basename):
    """
    Extract Hs from names like 'Job-H-0_6_2300_30' or 'Job-H-0_6-1950_2'.
    Returns float or default 0.6 if not found.
    """
    m = re.search(r'Job\-H\-([0-9]_[0-9])', job_basename)
    if m:
        return float(m.group(1).replace('_', '.'))
    return 0.6

def read_uplift_df(csv_path):
    """
    Read per-job CSV (Job, Step, FrameIndex, Time, uplift_B-..., sliding_B-...).
    Return (df_uplift, time_col, block_cols) where df_uplift has columns:
      time_col='Time', and block columns named 'B-...' (values are uplifts).
    """
    df = pd.read_csv(csv_path)
    # Identify time column
    time_col = 'Time' if 'Time' in df.columns else df.columns[0]
    # Keep only uplift columns and rename: uplift_B-1-1 -> B-1-1
    uplift_cols = [c for c in df.columns if c.startswith('uplift_B-')]
    rename_map = dict((c, c.replace('uplift_', '')) for c in uplift_cols)
    df_uplift = df[[time_col] + uplift_cols].copy()
    df_uplift.rename(columns=rename_map, inplace=True)
    block_cols = [c.replace('uplift_', '') for c in uplift_cols]
    return df_uplift, time_col, block_cols

def nearest_time_row(df_uplift, time_col, target):
    """Return the row (as Series) whose time is closest to target."""
    tvals = df_uplift[time_col].values.astype(float)
    idx = int(np.argmin(np.abs(tvals - float(target))))
    return df_uplift.iloc[idx]

def simulate_residual_profiles(Hs):
    """
    Build residual profiles for 5 extreme waves (Rayleigh top 1..5 in 1000),
    return (residual_profiles, impact_centers, wave_names).
    """
    sigma = Hs / np.sqrt(2.0)
    exceed_probs = np.arange(1, 6, 1) / 1000.0
    wave_heights = sigma * np.sqrt(-2.0 * np.log(exceed_probs))

    residual_profiles = {}
    impact_centers = {}
    wave_names = []

    for i, H_wave in enumerate(wave_heights):
        Tm_1_0 = Tp / 1.1
        L_m_1_0 = (g * (Tm_1_0 ** 2)) / (2.0 * np.pi)
        xi = tan_beta / np.sqrt(H_wave / L_m_1_0)
        if xi < 3.0:
            z_SWL = H_wave * (0.8 + 0.6 * np.tanh(xi - 2.1))
            x_center = (SWL - z_SWL) / tan_beta
        else:
            # No impact (as in your earlier logic)
            x_center = (SWL - 0.0) / tan_beta

        w_base = (7.0 / 6.0) * H_wave
        w_top  = (1.0 / 3.0) * H_wave
        x_left_base  = x_center - w_base / 2.0
        x_right_base = x_center + w_base / 2.0
        x_left_top   = x_center - w_top  / 2.0
        x_right_top  = x_center + w_top  / 2.0

        denom = (xi - 0.2) ** 2
        P_dimless = 8.0 - 1.6 * xi - 2.0 / denom
        P_max = P_dimless * rho_w * g * H_wave

        p_vals = np.zeros_like(x)
        # Trapezoid (only if meaningful span)
        for j, xx in enumerate(x):
            if x_left_base <= xx < x_left_top and (x_left_top != x_left_base):
                p_vals[j] = P_max * (xx - x_left_base) / (x_left_top - x_left_base)
            elif x_left_top <= xx <= x_right_top:
                p_vals[j] = P_max
            elif x_right_top < xx <= x_right_base and (x_right_base != x_right_top):
                p_vals[j] = P_max * (x_right_base - xx) / (x_right_base - x_right_top)

        # Solve filter response
        b_vec = -p_vals.copy()
        b_vec[0]  = 0.0
        b_vec[-1] = 0.0

        A = np.zeros((N, N))
        coeff = (Lambda ** 2) / (dx ** 2)
        for jj in range(1, N - 1):
            A[jj, jj - 1] = coeff
            A[jj, jj]     = -2.0 * coeff - 1.0
            A[jj, jj + 1] = coeff
        A[0, 0]   = 1.0
        A[-1, -1] = 1.0

        phi_F = solve(A, b_vec)
        residual = phi_F - p_vals

        wave_name = "Wave %d in 1000, H=%.2f" % (i + 1, H_wave)
        residual_profiles[wave_name] = residual
        impact_centers[wave_name] = x_center
        wave_names.append(wave_name)

    return residual_profiles, impact_centers, wave_names

def wave_index_for_time(t):
    """Map time to active wave index based on gravity_time and pulse_interval."""
    for wave_idx in range(total_waves):
        t_start = gravity_time + wave_idx * pulse_interval
        t_end = gravity_time + (wave_idx + 1) * pulse_interval
        if (t >= t_start) and (t < t_end):
            return wave_idx
    return None

def plot_one(csv_path):
    # ---- Read and reshape data
    df_uplift, time_col, block_cols = read_uplift_df(csv_path)

    # ---- Choose the time slice closest to target_time
    row = nearest_time_row(df_uplift, time_col, target_time)
    t_sel = float(row[time_col])

    # ---- Pull the single-time block values as a Series, columns 'B-...'
    block_data = row[block_cols]

    # ---- Extract Hs from job name
    job_basename = os.path.splitext(os.path.basename(csv_path))[0]
    # Trim suffix to recover job name for title
    job_name_for_title = job_basename.replace(CSV_SUFFIX.replace('.csv',''), '')
    Hs = extract_hs_from_job(job_basename)

    # ---- Simulate residual profiles for that Hs (as in your script)
    residual_profiles, impact_centers, wave_names = simulate_residual_profiles(Hs)

    # Determine which wave is active for t_sel
    active_idx = wave_index_for_time(t_sel)
    if active_idx is not None:
        wave_name = wave_names[active_idx]
    else:
        wave_name = wave_names[0]  # fallback

    residual_vals = residual_profiles[wave_name]
    x_center = impact_centers[wave_name]
    residual_vals_kPa = residual_vals / 1000.0

    # ---- Begin plotting
    fig = plt.figure(figsize=(10, 13))
    gs = gridspec.GridSpec(
        nrows=2, ncols=2,
        width_ratios=[3, 1], height_ratios=[6, 1],
        wspace=0.05, hspace=0.3
    )

    # --- Top left: block grid
    ax_grid = fig.add_subplot(gs[0, 0])

    for row_num in range(1, rows + 1):
        current_y = (rows - row_num) * (full_h + v_spacing)
        is_half = (row_num % 2 == 1)
        if is_half:
            block_sequence = ([(half_w, "B-%d-1" % row_num)] +
                              [(full_w, "B-%d-%d" % (row_num, col)) for col in range(2, 10)] +
                              [(half_w, "B-%d-10" % row_num)])
        else:
            block_sequence = [(full_w, "B-%d-%d" % (row_num, col)) for col in range(1, 10)]

        x_pos = 0.0
        for block_w, block_name in block_sequence:
            if block_name in block_data.index:
                val = block_data[block_name]
                if pd.isnull(val):
                    color = no_data_color
                else:
                    color = cmap(norm(val))
            else:
                color = no_data_color

            rect = patches.Rectangle((x_pos, current_y),
                                     block_w, full_h,
                                     edgecolor='black',
                                     facecolor=color,
                                     linewidth=1)
            ax_grid.add_patch(rect)
            x_pos += block_w + horizontal_spacing

    ax_grid.set_aspect('equal')
    ax_grid.set_xlim(0, grid_width)
    ax_grid.set_ylim(0, rows * full_h)
    ax_grid.axis('off')

    # Colorbar on the left
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cax = fig.add_axes([-0.02, 0.5, 0.02, 0.4])
    cbar = plt.colorbar(sm, cax=cax, orientation='vertical')
    cbar.set_label('Relative Uplift [m]')
    cbar.ax.yaxis.set_label_position('left')
    cbar.ax.yaxis.set_ticks_position('left')

    # --- Top right: residual pressure profile
    ax_side = fig.add_subplot(gs[0, 1])
    ax_side.plot(residual_vals_kPa, x, color='#1f77b4', linewidth=2, label="Residual Pressure\n%s" % wave_name)
    ax_side.axhline(x_mwl, color='darkblue', linestyle='-', linewidth=2, label='MWL')
    ax_side.axhline(x_center, color='black', linestyle='--', linewidth=2, label='Impact Center')
    ax_side.invert_yaxis()
    ax_side.set_xlabel('Pressure [kPa]')
    ax_side.set_ylabel('x along slope [m]')
    ax_side.set_ylim(5.5, 16.5)
    ax_side.yaxis.tick_right()
    ax_side.yaxis.set_label_position("right")
    ax_side.set_title("Residual Pressure Profile")
    ax_side.grid(True)
    ax_side.legend(loc='lower right', fontsize=8)

    # --- Bottom: max uplift over time
    ax_uplift = fig.add_subplot(gs[1, :])

    # Clean fly-out as NaN, then max over blocks each time row
    block_df_all = df_uplift[block_cols].copy()
    fly_out_mask = (block_df_all == fly_out_value)
    block_df_clean = block_df_all.where(~fly_out_mask, np.nan)
    max_uplift = block_df_clean.max(axis=1)

    # Mark first loss per block by looking for first fly_out occurrence
    first_loss_times = []
    for col in block_cols:
        idxs = np.where(fly_out_mask[col].values)[0].tolist()
        if idxs:
            first_loss_times.append(df_uplift[time_col].iloc[idxs[0]])
    first_loss_times = sorted(set(first_loss_times))

    ax_uplift.plot(df_uplift[time_col], max_uplift, color='navy', linewidth=2, label='Max Uplift')
    ax_uplift.axvline(t_sel, color='red', linestyle='-', linewidth=3, label="t = %.2f s" % t_sel)

    if len(first_loss_times) > 0:
        cross_y = (np.nanmax(max_uplift.values) * 1.1) if np.nanmax(max_uplift.values) > 0 else 0.01
        ax_uplift.scatter(first_loss_times, [cross_y] * len(first_loss_times),
                          color='red', marker='x', s=80, label='Block Loss (First Occurrence)')

    ax_uplift.set_xlabel("Time [s]")
    ax_uplift.set_xlim(0.0, max(df_uplift[time_col].max(), t_sel) * 1.01)
    ax_uplift.set_ylabel("Max Relative Uplift [m]")
    ax_uplift.set_title("Maximum Relative Uplift Over Time (All Blocks)")
    ax_uplift.legend(loc='upper right', fontsize=8)
    ax_uplift.grid(True)

    # --- Global title
    fig.suptitle("Job: %s   |   Hs = %.2f m   |   Time = %.2f s" %
                 (job_name_for_title, Hs, t_sel),
                 fontsize=16, fontweight="bold")

    plt.subplots_adjust(left=0, right=1, top=0.95, bottom=0, wspace=0, hspace=0)

    # Save PNG next to CSV
    out_png = csv_path.replace('.csv', '_viz.png')
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    print("Saved:", out_png)
    plt.close(fig)

# -----------------------------
# Run for all matching CSVs
# -----------------------------
def main():
    files = [f for f in os.listdir(SCRIPT_DIR) if f.endswith(CSV_SUFFIX)]
    if not files:
        print("No per-job CSVs found in: %s" % SCRIPT_DIR)
        return
    for f in sorted(files):
        try:
            plot_one(os.path.join(SCRIPT_DIR, f))
        except Exception as e:
            print("Failed plotting for %s : %s" % (f, str(e)))

if __name__ == '__main__':
    main()


Grid width: 4.900 m
Saved: d:\tpennock\GitHub\Thesis_Repository\workfolder\uplift matrix 2\Job-H-0_8_2300_30_uplift_downdrift_ALLSTEPS_ALLFRAMES_viz.png
Saved: d:\tpennock\GitHub\Thesis_Repository\workfolder\uplift matrix 2\Job-H-0_9_2300_30_uplift_downdrift_ALLSTEPS_ALLFRAMES_viz.png
Saved: d:\tpennock\GitHub\Thesis_Repository\workfolder\uplift matrix 2\Job-H-1_0_2300_30_uplift_downdrift_ALLSTEPS_ALLFRAMES_viz.png
Saved: d:\tpennock\GitHub\Thesis_Repository\workfolder\uplift matrix 2\all_jobs_uplift_downdrift_ALLSTEPS_ALLFRAMES_viz.png
